# Train Detector

This notebook trains a YOLOv8-based dice detector using KerasCV.

Goals:
- load train and validation splits
- build the input pipeline for object detection
- train a single-class detector for dice localization
- save the best model and training logs

In [ ]:
import json
from datetime import datetime

import numpy as np
import tensorflow as tf
import keras
import keras_cv
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from project_config import (
    TRAIN_JSON,
    VAL_JSON,
    MODELS_DIR,
    SEED,
    AUTOTUNE,
    DETECTOR_CLASS_NAME,
    NUM_DETECTOR_CLASSES,
    BOUNDING_BOX_FORMAT,
    DETECTOR_BATCH_SIZE,
    DETECTOR_TARGET_SIZE,
    DETECTOR_LEARNING_RATE,
    DETECTOR_EPOCHS,
    setup_reproducibility,
)
from utils.detector_data import (
    validate_and_fix_sample,
    build_ragged_arrays,
    load_image,
    load_example,
    filter_nonempty_batch,
    dict_to_tuple,
)

In [ ]:
# Optional workaround for environments with broken SSL certificate validation.
import os
import requests
import urllib3

os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

_old_merge_environment_settings = requests.Session.merge_environment_settings

def _patched_merge_environment_settings(self, url, proxies, stream, verify, cert):
    settings = _old_merge_environment_settings(self, url, proxies, stream, verify, cert)
    settings["verify"] = False
    return settings

requests.Session.merge_environment_settings = _patched_merge_environment_settings

In [ ]:
setup_reproducibility(SEED)

print("TensorFlow:", tf.__version__)
print("KerasCV:", keras_cv.__version__)
print("Keras:", keras.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_data = json.load(f)

print("Train samples:", len(train_data))
print("Val samples:", len(val_data))

if not train_data:
    raise ValueError(f"Training split is empty: {TRAIN_JSON}")
if not val_data:
    raise ValueError(f"Validation split is empty: {VAL_JSON}")

In [ ]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
BEST_MODEL_PATH = MODELS_DIR / f"dice_detector_best_{RUN_ID}.keras"
TRAINING_CSV_PATH = MODELS_DIR / f"dice_detector_training_{RUN_ID}.csv"

print("BEST_MODEL_PATH:", BEST_MODEL_PATH)
print("TRAINING_CSV_PATH:", TRAINING_CSV_PATH)

In [ ]:
train_image_paths, train_boxes_rt, train_classes_rt = build_ragged_arrays(train_data)
val_image_paths, val_boxes_rt, val_classes_rt = build_ragged_arrays(val_data)

print("Ragged train boxes shape:", train_boxes_rt.bounding_shape())
print("Ragged val boxes shape:", val_boxes_rt.bounding_shape())

In [ ]:
resize_layer = keras_cv.layers.Resizing(
    DETECTOR_TARGET_SIZE,
    DETECTOR_TARGET_SIZE,
    bounding_box_format=BOUNDING_BOX_FORMAT,
    pad_to_aspect_ratio=True,
)

spatial_augmenter = keras_cv.layers.RandomFlip(
    mode="horizontal",
    bounding_box_format=BOUNDING_BOX_FORMAT,
)

def apply_train_preprocessing(inputs):
    inputs = resize_layer(inputs)
    inputs["images"] = inputs["images"] / 255.0
    inputs = spatial_augmenter(inputs)
    return inputs

def apply_eval_preprocessing(inputs):
    inputs = resize_layer(inputs)
    inputs["images"] = inputs["images"] / 255.0
    return inputs

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_image_paths, train_classes_rt, train_boxes_rt)
)
val_ds = tf.data.Dataset.from_tensor_slices(
    (val_image_paths, val_classes_rt, val_boxes_rt)
)

train_ds = train_ds.shuffle(len(train_data), seed=SEED, reshuffle_each_iteration=True)
train_ds = train_ds.map(load_example, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.ragged_batch(DETECTOR_BATCH_SIZE, drop_remainder=False)
train_ds = train_ds.map(apply_train_preprocessing, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.filter(filter_nonempty_batch)
train_ds = train_ds.map(dict_to_tuple, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.repeat().prefetch(AUTOTUNE)

val_ds = val_ds.map(load_example, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.ragged_batch(DETECTOR_BATCH_SIZE, drop_remainder=False)
val_ds = val_ds.map(apply_eval_preprocessing, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.filter(filter_nonempty_batch)
val_ds = val_ds.map(dict_to_tuple, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.repeat().prefetch(AUTOTUNE)

steps_per_epoch = int(np.ceil(len(train_data) / DETECTOR_BATCH_SIZE))
validation_steps = int(np.ceil(len(val_data) / DETECTOR_BATCH_SIZE))

print("steps_per_epoch:", steps_per_epoch)
print("validation_steps:", validation_steps)

In [ ]:
batch_images, batch_targets = next(iter(train_ds))
print("images:", batch_images.shape, batch_images.dtype)
print("boxes bounding shape:", batch_targets["boxes"].bounding_shape())
print("classes bounding shape:", batch_targets["classes"].bounding_shape())
print("box row lengths:", batch_targets["boxes"].row_lengths().numpy())
print("class row lengths:", batch_targets["classes"].row_lengths().numpy())
print("unique detector classes in batch:", np.unique(batch_targets["classes"].flat_values.numpy()))

assert np.any(batch_targets["boxes"].row_lengths().numpy() > 0), "Batch has no boxes after preprocessing."

In [ ]:
def visualize_batch(dataset, max_images=2):
    images, targets = next(iter(dataset))
    boxes = targets["boxes"]

    max_images = min(max_images, images.shape[0])
    fig, axes = plt.subplots(1, max_images, figsize=(7 * max_images, 7))
    if max_images == 1:
        axes = [axes]

    for i in range(max_images):
        ax = axes[i]
        image = images[i].numpy()
        ax.imshow(np.clip(image, 0.0, 1.0))

        sample_boxes = boxes[i].numpy()
        for box in sample_boxes:
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)

        ax.axis("off")

    plt.tight_layout()
    plt.show()

# visualize_batch(train_ds, max_images=2)

In [ ]:
tf.keras.backend.clear_session()

backbone = keras_cv.models.YOLOV8Backbone.from_preset(
    "yolo_v8_xs_backbone_coco",
    load_weights=True,
)

model = keras_cv.models.YOLOV8Detector(
    num_classes=NUM_DETECTOR_CLASSES,
    bounding_box_format=BOUNDING_BOX_FORMAT,
    backbone=backbone,
    fpn_depth=1,
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=DETECTOR_LEARNING_RATE,
    global_clipnorm=10.0,
)

model.compile(
    optimizer=optimizer,
    classification_loss="binary_crossentropy",
    box_loss="ciou",
    jit_compile=False,
)

model.prediction_decoder = keras_cv.layers.NonMaxSuppression(
    bounding_box_format=BOUNDING_BOX_FORMAT,
    from_logits=False,
    confidence_threshold=0.2,
    iou_threshold=0.5,
)

print("YOLOv8 detector ready.")

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        min_delta=1e-4,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(TRAINING_CSV_PATH)),
    keras.callbacks.TerminateOnNaN(),
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=DETECTOR_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1,
)

print("Training finished.")
print("Best model saved to:", BEST_MODEL_PATH)
print("CSV log saved to:", TRAINING_CSV_PATH)